# MARS-X quick start: bring your own model

Train or load a navigation policy, export it as ONNX `policy/1`, evaluate it on the `jezero-v1` suite
(real Jezero terrain, Mars conditions) and get a `run.json` you can replay in 3D in the MARS-X web app.

> Simulation only. Not affiliated with or endorsed by NASA, JPL or Caltech.

## 1. Install (pin a commit for reproducibility)

In [ ]:
COMMIT = "main"  # replace with a commit SHA to reproduce exactly the same scenarios, suite and scores
REPO = "Atsushi2036/mars-x"
!pip install -q "marsx[train] @ git+https://github.com/{REPO}@{COMMIT}#subdirectory=python"

## 2. Get the data

Downloads the processed Jezero dataset from the GitHub Release recorded in `data/dataset.lock.json` and checks its
sha256. If no release is published at this commit, the notebook falls back to a small synthetic map (the API is the
same; scores are not comparable).

In [ ]:
import hashlib, json, os, tarfile, urllib.request
from pathlib import Path

def get(url, timeout=600):
    with urllib.request.urlopen(url, timeout=timeout) as r:  # HTTPS with certificate verification
        return r.read()

raw = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}"
DATA = Path("/content/jezero-v1") if Path("/content").exists() else Path("jezero-v1")
TERRAIN = "synthetic"
try:
    lock = json.loads(get(f"{raw}/data/dataset.lock.json"))["jezero-v1"]
    blob = get(lock["url"])
    assert hashlib.sha256(blob).hexdigest() == lock["sha256"], "dataset sha256 mismatch"
    archive = DATA.parent / "jezero-v1.tar.gz"
    archive.write_bytes(blob)
    with tarfile.open(archive) as tf:
        tf.extractall(DATA.parent, filter="data")
    os.environ["MARSX_DATA"] = str(DATA)
    TERRAIN = str(DATA)
    print("dataset ready:", DATA)
except Exception as exc:  # no release yet / offline
    print("using the synthetic map:", type(exc).__name__, exc)
SUITE = Path("suite.json")
try:
    SUITE.write_bytes(get(f"{raw}/bench/jezero-v1/suite.json"))
except Exception as exc:
    print("suite not available:", exc)

## 3. The environment (Gymnasium) — single and vectorised

In [ ]:
import time
import numpy as np
import gymnasium as gym
from marsx.env import scenarios
print(scenarios.ids())

env = gym.make("marsx.env:MarsX/Jezero-v1", scenario="jezero-v1/l1/rtg-geometry", terrain=TERRAIN)
obs, info = env.reset(seed=0)
print(obs.shape, info["cell"])  # obs/1: float32[505]; act/1: Discrete(9) = N S E W NE NW SE SW WAIT

vec = gym.make_vec("marsx.env:MarsX/Jezero-v1", num_envs=4096, vectorization_mode="vector_entry_point",
                   terrain=TERRAIN)
vec.reset(seed=0)
t = time.perf_counter()
for _ in range(50):
    vec.step(vec.action_space.sample())
print(f"{4096 * 50 / (time.perf_counter() - t):,.0f} agent-steps/s in one process")

## 4. Train a small PPO (CPU)

Uses the NumPy-vectorised engine through `SB3VecAdapter` (no subprocesses). A few hundred thousand steps already
learn short routes; the reference policy in `models/jezero-v1/` used millions of steps.

In [ ]:
from stable_baselines3 import PPO
from marsx.env._build import resolve_terrain
from marsx.env.core import NavBatch
from marsx.sb3 import SB3VecAdapter

terrain = resolve_terrain(TERRAIN)
engine = NavBatch(terrain, scenarios.get("jezero-v1/l1/rtg-geometry"), num_envs=64, seed=0, r_min=10, r_max=60)
model = PPO("MlpPolicy", SB3VecAdapter(engine), n_steps=256, batch_size=4096, n_epochs=4,
            policy_kwargs={"net_arch": [128, 128]}, device="cpu", seed=0, verbose=0)
model.learn(300_000)

## 5. Export ONNX `policy/1` and validate it (same rules as the web loader)

In [ ]:
from marsx.train import export_onnx
export_onnx.export_logits(model.policy, "policy.onnx")   # obs float32[N,505] -> logits float32[N,9], Gemm/Tanh only
print(export_onnx.validate("policy.onnx"))
card = export_onnx.policy_card("policy.onnx", name="my-team/ppo", version="0.0.1",
                               intended_use="MARS-X L1 navigation experiment.", limitations="Short training run.")
Path("policy-card.json").write_text(card.model_dump_json(by_alias=True, indent=2, exclude_none=True))

## 6. Evaluate on the suite and build the scorecard

Needs the real dataset (step 2). `marsx scorecard` re-simulates every episode to verify the run.

In [ ]:
if TERRAIN != "synthetic" and SUITE.exists():
    !marsx evaluate --policy policy.onnx --level L1 --suite suite.json --name my-team/ppo --out run_L1.json
    !PYTHONIOENCODING=utf-8 marsx scorecard run_L1.json --suite suite.json --out scorecard.json
else:
    print("skipped: needs the released dataset and suite.json")

## 7. Replay in 3D

Download `run_L1.json` and drop it on the MARS-X web app (Replay panel). It is validated in a worker and replayed on
the Jezero terrain; nothing leaves your browser. You can also drag `policy.onnx` onto the app to run it live.

In [ ]:
try:
    from google.colab import files
    for f in ("run_L1.json", "scorecard.json", "policy.onnx", "policy-card.json"):
        if Path(f).exists():
            files.download(f)
except ImportError:
    print("not in Colab: files are in", Path.cwd())